# Working with Tabular Data in `pandas`

`01_Python_logic_intro.ipynb` covered variables, `if`, and `for`. This notebook covers **pandas**, the library every app in this repo uses to hold measurement data once it's been pulled out of NOMAD -- rows of samples/curves, columns of properties.

`04_HandlingJVdata.ipynb` leans on everything here (`.groupby`, boolean masks, building a DataFrame from a list of dicts), so it's worth doing this one first if pandas is new to you. No NOMAD login needed -- everything below uses made-up data.

## What You'll Learn

This notebook covers:

1. **DataFrames** -- building one from a list of dicts, the same shape NOMAD API responses get flattened into
2. **Selecting data** -- columns, rows, and boolean masks
3. **Adding/modifying columns** -- vectorized operations and `.str` methods
4. **`groupby`** -- summarizing and looping over groups
5. **A few everyday methods** -- `.sort_values`, `.describe`, `.value_counts`


## 0. Running code cells

- Run a cell with **Shift + Enter**
- If you get `NameError: name 'pd' is not defined`, run the import cell below first.


In [ ]:
import pandas as pd

print("pandas", pd.__version__)


## 1. What is a DataFrame?

A DataFrame is a table: rows and named columns. The easiest way to build one from scratch is a **list of dicts** -- one dict per row -- which is exactly the shape `04_HandlingJVdata.ipynb` builds from NOMAD's API responses.


In [ ]:
rows = [
    {"lab_id": "S1", "condition": "0.5mg/mL", "direction": "Reverse", "PCE(%)": 18.2},
    {"lab_id": "S1", "condition": "0.5mg/mL", "direction": "Forward", "PCE(%)": 17.1},
    {"lab_id": "S2", "condition": "0.5mg/mL", "direction": "Reverse", "PCE(%)": 19.0},
    {"lab_id": "S3", "condition": "1.0mg/mL", "direction": "Reverse", "PCE(%)": 20.4},
    {"lab_id": "S3", "condition": "1.0mg/mL", "direction": "Forward", "PCE(%)": 19.8},
    {"lab_id": "S4", "condition": "1.0mg/mL", "direction": "Reverse", "PCE(%)": 21.1},
]

df = pd.DataFrame(rows)
df


In [ ]:
print("shape (rows, columns):", df.shape)
print("\ncolumn names:", list(df.columns))
print("\ndtypes:")
print(df.dtypes)


## 2. Selecting data

- One column: `df["col"]` (returns a **Series**)
- Several columns: `df[["a", "b"]]` (returns a **DataFrame** -- note the double brackets)
- Rows by position: `df.iloc[0]`
- Rows by a **condition** (a boolean mask) -- this is the one you'll use constantly


In [ ]:
print(df["PCE(%)"])


In [ ]:
df[["lab_id", "PCE(%)"]]


In [ ]:
# Boolean mask: df["PCE(%)"] > 19 is a column of True/False,
# used to filter rows the same way an `if` filters values in a loop.
high_pce = df[df["PCE(%)"] > 19]
high_pce


In [ ]:
# Combine conditions with & (and) / | (or) -- note the parentheses, they're required
df[(df["direction"] == "Reverse") & (df["PCE(%)"] > 19)]


## 3. Adding and modifying columns

Operations on a column apply to every row at once (no `for` loop needed) -- this is called a **vectorized** operation.


In [ ]:
df["PCE(fraction)"] = df["PCE(%)"] / 100
df


In [ ]:
# .str gives you string methods on a whole column at once
df["condition_clean"] = df["condition"].str.replace("mg/mL", "")
df[["condition", "condition_clean"]]


## 4. Grouping and summarizing: `groupby`

`groupby` splits a DataFrame into groups sharing the same value in one column, so you can summarize (`.median()`, `.mean()`, `.count()`, ...) or loop over each group individually.


In [ ]:
df.groupby("condition")["PCE(%)"].median()


In [ ]:
# Looping over groups -- this exact pattern picks out "the representative device
# per condition" in 04_HandlingJVdata.ipynb
for condition, group in df.groupby("condition"):
    med = group["PCE(%)"].median()
    closest = (group["PCE(%)"] - med).abs().idxmin()
    print(f"{condition}: median={med:.1f}%, closest row index={closest}")


## 5. A few more useful methods

- `df.sort_values("col")` -- sort by a column
- `df.describe()` -- quick stats (mean, std, min/max, ...) for every numeric column
- `df["col"].value_counts()` -- how many rows have each value
- `df.dropna()` -- drop rows with missing values


In [ ]:
df.sort_values("PCE(%)", ascending=False)


In [ ]:
df.describe()


In [ ]:
df["condition"].value_counts()


## Exercises (recommended)

1. Add a row for a new sample `"S5"` and re-run the `groupby` median cell.
2. Filter `df` to rows where `direction == "Forward"` **and** `PCE(%) < 20`.
3. Add a column `"PCE(rounded)"` that rounds `PCE(%)` to the nearest integer (hint: `.round()`).
4. In `04_HandlingJVdata.ipynb`'s row-building cell, find the line that turns `rows` (a list of dicts) into `df_exp` -- it's the same one-liner as cell 4 above.
